# Neon RAG benchmark matrix

This notebook plans and runs the 27 combinations. Full ingestion is opt-in: Vidore has native qrels; ParseBench and GovDocs require supplied/generated queries. The first cell is a deterministic offline smoke path.

In [ ]:
from dataclasses import replace
from neon_rag_benchmarks.config import BenchmarkConfig, MODELS
from neon_rag_benchmarks.matrix import matrix
from neon_rag_benchmarks.pipeline import run_matrix

# Parameter controls: set SMOKE=False for bounded live HF + Neon + gateway execution.
config = BenchmarkConfig.from_env()
SMOKE = True
DATASET = 'all'  # or vidore, parsebench, govdocs
EMBEDDING_MODEL = 'all'  # or nomic, minilm, mpnet
TOP_K = config.top_k
EXACT_SCAN = True  # exact scan is always recorded for live runs
config = replace(config, top_k=TOP_K)
dataset_names = None if DATASET == 'all' else (DATASET,)
model_keys = None if EMBEDDING_MODEL == 'all' else (EMBEDDING_MODEL,)
print('controls:', DATASET, EMBEDDING_MODEL, TOP_K, EXACT_SCAN)

In [ ]:
# This is the executable workflow for both modes; live mode validates gateway config first.
results = run_matrix(config, smoke=SMOKE, persist=False, dataset_names=dataset_names, model_keys=model_keys)
print(f'{len(matrix())} full matrix combinations; {len(results)} endpoint records in this run')
print(results[0])

## Live execution contract

Use `neon_rag_benchmarks.embeddings.embed_texts`/`embed_query`, `db.setup`/`db.search`, and `gateway.answer`. Time each phase. For Vidore load corpus/test markdown, queries/test query, and qrels/test query_id/corpus_id/score. ParseBench uses `text_content` and GovDocs needs PDF text extraction plus `broken_pdf=false`; both need user-provided or generated queries. Answer results must retain the endpoint environment variable label.